In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc 
import anndata as ad
import pertpy as pt
import os

In [ ]:
# Set working path 
base_path = '/home/EOCRC_atlas/'

In [ ]:
# Load adata object
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_processed_withTier2annotation.h5ad'))

In [ ]:
# Subset to just MSS samples 
print(adata.shape)
adata = adata[adata.obs['MSI_v2']=="MSS: STABLE"].copy()
print(adata.shape)

In [ ]:
# Add columns for age that are scaled by zscore and minmax 
# zscore 
adata.obs['Age'] = pd.to_numeric(adata.obs['Age'], errors='coerce')
adata.obs['age_scaled'] = (adata.obs['Age'] - adata.obs['Age'].mean()) / adata.obs['Age'].std()

# min-max scale
adata.obs["age_minmax"] = (adata.obs["Age"] - adata.obs["Age"].min()) / (adata.obs["Age"].max() - adata.obs["Age"].min())

In [ ]:
# Subset to non-mixed marker cell types for compositional analysis 
run_celltypes = ['Adipocytes', 'B cell', 'CD4 T cells', 'CD8 T cells',
                  'CEACAM1 colonocyte-like', 'Cycing endothelium', 'Cycling Myeloid',
                  'Cycling Stromal', 'Cycling T cells', 'Cycling plasma cell', 'DC',
                  'Enteroendocrine-like', 'Fibroblast', 'Fibroblast-BMP5-SOX6',
                  'Fibroblast-C3', 'Fibroblast-Infl', 'Fibroblast-KCNN3',
                  'Fibroblast-MMP2-THY1', 'Germinal center / Cycling B cell',
                  'Glial cells', 'HSP-hi - B cell', 'HSP-hi Myeloid',
                  'HSP-hi Stromal', 'HSP-hi T cells', 'HSP-hi glial', 'ILCs',
                  'LGR5 stem cell-like', 'Lymphatic endothelium',
                  'MT-Ribo-hi Myeloid', 'MT-Ribo-hi Stromal', 'MT-Ribo-hi T cells',
                  'MT-Ribo-hi endothelium', 'MT-Ribo-hi epithelial',
                  'MUC2 goblet-like', 'Macrophage-Monocyte', 'Mast',
                  'Myofibroblast-SMC', 'NK-Cytotoxic T cells', 'Neuronal cells',
                  'Neutrophil', 'Patient-specific', 'Pericytes', 'Plasma cell',
                  'Regulatory T cells', 'T helper cells', 'Vascular endothelium']

adata = adata[adata.obs['Annotation_Tier2'].isin(run_celltypes)].copy()

In [ ]:
# Check for unknown variables in clinical covariates of interest 
print(np.unique(adata.obs['Sidedness']))
print(np.unique(adata.obs['Therapy_v2']))
print(np.unique(adata.obs['Sex']))
print(np.unique(adata.obs['Overall_Stage']))

In [ ]:
# setup object for sccoda - pull in any coviariates we may want to test 
sccoda_data = pt.tl.Sccoda()
adata_coda = sccoda_data.load(
    adata,
    type="cell_level",
    generate_sample_level=True,
    cell_type_identifier="Annotation_Tier1",
    sample_identifier="FRID",
    covariate_obs=["age_scaled", "age_minmax", "Sidedness", "Therapy_v2", "Sex", "Overall_Stage", "Cohort"]
)

In [ ]:
# prepare object and run sccoda object - stromal cells as reference 
sccoda_data.prepare(adata_coda, formula="age_minmax + Sidedness + Therapy_v2 + Sex + Overall_Stage", reference_cell_type="Stromal")
sccoda_data.run_nuts(adata_coda,num_samples=10000,num_warmup=1000)
sccoda_data.get_effect_df(adata_coda)

In [ ]:
# prepare object and run sccoda object - endothelial cells as reference 
sccoda_data.prepare(adata_coda, formula="age_minmax + Sidedness + Therapy_v2 + Sex + Overall_Stage", reference_cell_type="Endothelial")
sccoda_data.run_nuts(adata_coda,num_samples=10000,num_warmup=1000)
sccoda_data.get_effect_df(adata_coda)

In [ ]:
# prepare object and run sccoda object - myeloid cells as reference 
sccoda_data.prepare(adata_coda, formula="age_minmax + Sidedness + Therapy_v2 + Sex + Overall_Stage", reference_cell_type="Myeloid")
sccoda_data.run_nuts(adata_coda,num_samples=10000,num_warmup=1000)
sccoda_data.get_effect_df(adata_coda)

In [ ]:
# prepare object and run sccoda object - T cells as reference 
sccoda_data.prepare(adata_coda, formula="age_minmax + Sidedness + Therapy_v2 + Sex + Overall_Stage", reference_cell_type="T")
sccoda_data.run_nuts(adata_coda,num_samples=10000,num_warmup=1000)
sccoda_data.get_effect_df(adata_coda)

In [ ]:
# prepare object and run sccoda object - B cells as reference 
sccoda_data.prepare(adata_coda, formula="age_minmax + Sidedness + Therapy_v2 + Sex + Overall_Stage", reference_cell_type="B")
sccoda_data.run_nuts(adata_coda,num_samples=10000,num_warmup=1000)
sccoda_data.get_effect_df(adata_coda)

In [ ]:
# prepare object and run sccoda object - glial/neuronal cells as reference 
sccoda_data.prepare(adata_coda, formula="age_minmax + Sidedness + Therapy_v2 + Sex + Overall_Stage", reference_cell_type="Glial/Neuronal")
sccoda_data.run_nuts(adata_coda,num_samples=10000,num_warmup=1000)
sccoda_data.get_effect_df(adata_coda)

In [ ]:
# prepare object and run sccoda object - Epithelial
sccoda_data.prepare(adata_coda, formula="age_minmax + Sidedness + Therapy_v2 + Sex + Overall_Stage", reference_cell_type="Epithelial")
sccoda_data.run_nuts(adata_coda,num_samples=10000,num_warmup=1000)
sccoda_data.get_effect_df(adata_coda)